In [8]:

import pandas as pd
from pathlib import Path

root = Path.cwd()

if not (root / "data" / "processed").exists():
    root = root.parent

pluto = pd.read_csv(
    root / "data/processed/manhattan_pluto_26v2_cleaned.csv.gz",
    low_memory=False
)

pluto.head()

,borough,block,lot,landuse,lotarea,bldgarea,unitsres,unitstotal,yearbuilt,bbl,xcoord,ycoord,version,latitude,longitude,land_use_name,construction_period
0,MN,2066,59,5.0,4996.0,15000.0,0.0,2.0,1926.0,1020660059,999822.0,241356.0,26v2,40.829128,-73.943731,Commercial / office,1920–1939
1,MN,2065,7,6.0,7435.0,43621.0,0.0,1.0,1910.0,1020650007,999636.0,241041.0,26v2,40.828264,-73.944404,Industrial,1900–1919
2,MN,2065,12,3.0,4996.0,19740.0,26.0,26.0,1926.0,1020650012,999732.0,240988.0,26v2,40.828118,-73.944057,Multi-family elevator,1920–1939
3,MN,2065,21,2.0,2498.0,9330.0,10.0,10.0,1920.0,1020650021,999919.0,240885.0,26v2,40.827835,-73.943382,Multi-family walk-up,1920–1939
4,MN,2067,61,4.0,2175.0,8337.0,3.0,4.0,1926.0,1020670061,999834.0,241690.0,26v2,40.830045,-73.943687,Mixed residential / commercial,1920–1939


In [9]:

print("Number of rows:", len(pluto))
print("Columns:")
print(pluto.columns.tolist())


Number of rows: 42441
Columns:
['borough', 'block', 'lot', 'landuse', 'lotarea', 'bldgarea', 'unitsres', 'unitstotal', 'yearbuilt', 'bbl', 'xcoord', 'ycoord', 'version', 'latitude', 'longitude', 'land_use_name', 'construction_period']


In [10]:

for col in pluto.columns:
    if any(word in col.lower() for word in
           ["nta", "unitsres", "residential", "latitude", "longitude"]):
        print(col)


unitsres
latitude
longitude


In [11]:

import geopandas as gpd

nta = gpd.read_file(root / "data/raw/nynta2020.shp")

nta.head()


,BoroCode,BoroName,CountyFIPS,NTA2020,NTAName,NTAAbbrev,NTAType,CDTA2020,CDTAName,Shape_Leng,Shape_Area,geometry
0,3,Brooklyn,047,BK0101,Greenpoint,Grnpt,0,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),28919.560728,3.532181e+07,"POLYGON ((1003059.997 204572.025, 1002991.367 ..."
1,3,Brooklyn,047,BK0102,Williamsburg,Wllmsbrg,0,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),28134.082737,2.885285e+07,"POLYGON ((995851.916 203199.332, 995969.193 20..."
2,3,Brooklyn,047,BK0103,South Williamsburg,SWllmsbrg,0,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),18250.280091,1.520896e+07,"POLYGON ((998047.21 196303.325, 998157.901 196..."
3,3,Brooklyn,047,BK0104,East Williamsburg,EWllmsbrg,0,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),43184.800376,5.226741e+07,"POLYGON ((1005302.497 199455.73, 1005307.792 1..."
4,3,Brooklyn,047,BK0201,Brooklyn Heights,BkHts,0,BK02,BK02 Downtown Brooklyn-Fort Greene (CD 2 Appro...,14312.192285,9.982023e+06,"POLYGON ((986367.736 190549.239, 985813.836 19..."


In [13]:

print(nta.columns.tolist())
print("Coordinate system:", nta.crs)
print("Number of NTAs:", len(nta))


['BoroCode', 'BoroName', 'CountyFIPS', 'NTA2020', 'NTAName', 'NTAAbbrev', 'NTAType', 'CDTA2020', 'CDTAName', 'Shape_Leng', 'Shape_Area', 'geometry']
Coordinate system: EPSG:2263
Number of NTAs: 262


In [15]:

manhattan_nta = nta[nta["BoroCode"] == 1].copy()

print("Manhattan NTAs:", len(manhattan_nta))

manhattan_nta[["NTA2020", "NTAName"]].head()


Manhattan NTAs: 38


,NTA2020,NTAName
118,MN0101,Financial District-Battery Park City
119,MN0102,Tribeca-Civic Center
120,MN0191,The Battery-Governors Island-Ellis Island-Libe...
121,MN0201,SoHo-Little Italy-Hudson Square
122,MN0202,Greenwich Village


In [16]:

import geopandas as gpd

pluto["latitude"] = pd.to_numeric(pluto["latitude"], errors="coerce")
pluto["longitude"] = pd.to_numeric(pluto["longitude"], errors="coerce")

pluto_gdf = gpd.GeoDataFrame(
    pluto,
    geometry=gpd.points_from_xy(
        pluto["longitude"],
        pluto["latitude"]
    ),
    crs="EPSG:4326"
)

pluto_gdf = pluto_gdf.to_crs(manhattan_nta.crs)

print("PLUTO properties:", len(pluto_gdf))
print("Coordinate system:", pluto_gdf.crs)


PLUTO properties: 42441
Coordinate system: EPSG:2263


In [17]:

nta_boundaries = manhattan_nta[
    ["NTA2020", "NTAName", "geometry"]
].copy()

pluto_with_nta = gpd.sjoin(
    pluto_gdf,
    nta_boundaries,
    how="left",
    predicate="within"
)

print("Original PLUTO records:", len(pluto_gdf))
print("Records after join:", len(pluto_with_nta))
print("Properties matched:", pluto_with_nta["NTA2020"].notna().sum())
print("Properties unmatched:", pluto_with_nta["NTA2020"].isna().sum())

pluto_with_nta[["unitsres", "NTA2020", "NTAName"]].head()


Original PLUTO records: 42441
Records after join: 42441
Properties matched: 41688
Properties unmatched: 753


,unitsres,NTA2020,NTAName
0,0.0,MN0903,Hamilton Heights-Sugar Hill
1,0.0,MN0903,Hamilton Heights-Sugar Hill
2,26.0,MN0903,Hamilton Heights-Sugar Hill
3,10.0,MN0903,Hamilton Heights-Sugar Hill
4,3.0,MN0903,Hamilton Heights-Sugar Hill


In [18]:

matched = pluto_with_nta[
    pluto_with_nta["NTA2020"].notna()
].copy()

unmatched = pluto_with_nta[
    pluto_with_nta["NTA2020"].isna()
].copy()

print("Matched residential units:",
      matched["unitsres"].sum())

print("Unmatched residential units:",
      unmatched["unitsres"].sum())

print("Unmatched properties with residential units:",
      (unmatched["unitsres"] > 0).sum())

unmatched[["bbl", "unitsres", "latitude", "longitude"]].head(10)


Matched residential units: 955263.0
Unmatched residential units: 3693.0
Unmatched properties with residential units: 269


,bbl,unitsres,latitude,longitude
56,1020050027,0.0,40.82171,-73.959715
109,1015199041,0.0,NaN,NaN
161,1012439139,0.0,NaN,NaN
318,1011719145,0.0,NaN,NaN
319,1012809090,0.0,NaN,NaN
320,1014838110,0.0,NaN,NaN
321,1007028001,0.0,NaN,NaN
322,1005730001,0.0,NaN,NaN
324,1017741008,0.0,NaN,NaN
325,1007819010,0.0,NaN,NaN


In [19]:

pluto_by_nta = (
    matched.groupby(["NTA2020", "NTAName"], as_index=False)
    .agg(
        total_residential_units=("unitsres", "sum"),
        tax_lot_count=("bbl", "count")
    )
)

print("Number of NTAs:", len(pluto_by_nta))

pluto_by_nta.head(10)


Number of NTAs: 38


,NTA2020,NTAName,total_residential_units,tax_lot_count
0,MN0101,Financial District-Battery Park City,36436.0,554
1,MN0102,Tribeca-Civic Center,12249.0,866
2,MN0191,The Battery-Governors Island-Ellis Island-Libe...,0.0,9
3,MN0201,SoHo-Little Italy-Hudson Square,15323.0,1577
4,MN0202,Greenwich Village,21491.0,1081
5,MN0203,West Village,24535.0,2014
6,MN0301,Chinatown-Two Bridges,17754.0,1021
7,MN0302,Lower East Side,25685.0,921
8,MN0303,East Village,40193.0,2278
9,MN0401,Chelsea-Hudson Yards,46130.0,1923


In [20]:

missing_ntas = set(manhattan_nta["NTA2020"]) - set(pluto_by_nta["NTA2020"])

print("NTAs in PLUTO dataset:", len(pluto_by_nta))
print("NTAs missing from PLUTO:", missing_ntas)

print("Total matched residential units:",
      pluto_by_nta["total_residential_units"].sum())


NTAs in PLUTO dataset: 38
NTAs missing from PLUTO: set()
Total matched residential units: 955263.0


In [21]:

unmatched = pluto_with_nta[
    pluto_with_nta["NTA2020"].isna()
].copy()

print("Unmatched properties:", len(unmatched))

print("Residential units in unmatched properties:",
      unmatched["unitsres"].sum())

print("Unmatched properties with residential units:",
      (unmatched["unitsres"] > 0).sum())

print("Percentage of residential units unmatched:",
      round(
          unmatched["unitsres"].sum()
          / pluto["unitsres"].sum() * 100, 2
      ), "%")


Unmatched properties: 753
Residential units in unmatched properties: 3693.0
Unmatched properties with residential units: 269
Percentage of residential units unmatched: 0.39 %


In [22]:

output_path = root / "data/processed/manhattan_pluto_units_by_nta.csv"

pluto_by_nta.to_csv(output_path, index=False)

print("File saved successfully!")
print("Location:", output_path)
print("Number of neighborhoods:", len(pluto_by_nta))


File saved successfully!
Location: /Users/susanhuang/Documents/GitHub/manhattan-housing-affordability/data/processed/manhattan_pluto_units_by_nta.csv
Number of neighborhoods: 38


notes: 
- Total PLUTO records: 42,441 Manhattan tax lots
- Method: Spatially joined property coordinates to 2020 Manhattan NTA boundaries
- Matched records: 41,688 properties across all 38 Manhattan NTAs
- Unmatched records: 753 properties, representing 3,693 residential units (0.39% of total units)
- Aggregation: Summed residential units by NTA, totaling 955,263 matched units
- Limitation: Unmatched properties were excluded, potentially understating residential units in some neighborhoods